In [2]:
from json import JSONEncoder
import logging
import sys
import ndjson

# sys.path.append(r'../pickhardtpayments')
from pickhardtpayments.pickhardtpayments import *
# import pickhardtpayments.pickhardtpayments
# from pickhardtpayments.pickhardtpayments import ChannelGraph
# from pickhardtpayments.UncertaintyNetwork import UncertaintyNetwork
# from pickhardtpayments.OracleLightningNetwork import OracleLightningNetwork
# from pickhardtpayments.SyncPaymentSession import SyncPaymentSession
# from Payment import Payment

In [3]:
def set_logger():
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s.%(msecs)03d | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(logging.INFO)
    stdout_handler.setFormatter(formatter)
    file_handler = logging.FileHandler('pickhardt_pay.log', mode='w')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stdout_handler)

In [4]:
# subclass JSONEncoder
class PaymentEncoder(JSONEncoder):
    def default(self, o):
        return o.__dict__

In [5]:
set_logger()
logging.info('*** new payment simulation ***')

18:18:29.622 | INFO | *** new payment simulation ***


In [6]:
# *** Setup ***
graph = ChannelGraph("./pickhardtpayments/channels.sample.json")
#graph = ChannelGraph("./pickhardtpayments/listchannels20220412.json")

def only_channels_with_return_channels(_graph: ChannelGraph):
    channels_with_no_return_channel = []
    for edge in _graph.network.edges:
        if not _graph.network.has_edge(edge[1], edge[0]):
            channels_with_no_return_channel.append(edge)

    for edge in channels_with_no_return_channel:
        _graph.network.remove_edge(edge[0], edge[1], edge[2])

    if len(channels_with_no_return_channel) == 0:
        logging.debug("channel graph only had channels in both directions.")
    else:
        logging.debug("channel graph had unannounced channels.")
    return _graph

In [7]:
graph = only_channels_with_return_channels(graph)
uncertainty_network = UncertaintyNetwork(graph)
oracle_lightning_network = OracleLightningNetwork(graph)

In [8]:
# test of adjusted liquidity
set_liquidity = 100000
import random
chan = random.sample(list(oracle_lightning_network.network.edges), min(1000, len(oracle_lightning_network.network.edges)))
liquidity_test = True
for chan in random.sample(list(oracle_lightning_network.network.edges), 5):
    c = oracle_lightning_network.get_channel(chan[0], chan[1], chan[2])
    if c.actual_liquidity != set_liquidity:
        liquidity_test = False
        print(f"liquidity not {set_liquidity:,}")
if liquidity_test:
    print(f"liquidity in sample is {set_liquidity:,}")


liquidity in sample is 100,000


In [9]:
# create new payment session - will later be moved before payment loop to simulate sequence of payments
sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
print("Setup finished")

Setup finished


In [10]:
def get_possible_flows():
    c = 0
    possible_flow = list()
    break_at =  5
    for v_source in uncertainty_network.network.nodes():
        for v_destination in uncertainty_network.network.nodes():
            if v_source == v_destination:
                continue
            c += 1
            max_payable_amount = sim_session._oracle.theoretical_maximum_payable_amount(v_source, v_destination, 1000)
            flow = (v_source, v_destination, max_payable_amount)
            #flow = {"source": v_source, "model": v_destination,"max_flow": max_payable_amount}
            possible_flow.append(flow)
            if c == break_at: break
        if c == break_at: break
    return possible_flow

# maximum flow when assuming liquidity is capacity
maximum_flow = get_possible_flows()

# TODO: sample success rate for payment to find out how large a sample should be to be stable

# TODO: sample success rate for pickhardt payment to find out how large a sample should be to be stable

* save payment samples to json
* iterate over edges and find out "most prominent edges" in payment delivery